# Random Forest: Predicting Hard-Drive Failure
**IST 707 Final Project**

This notebook trains a random forest to predict whether a hard drive will fail in the next 30 days, using its SMART health readings from the last 30 days. It:

1. loads the training and calibration splits (the 2025 test split stays closed until final evaluation)
2. builds the model inputs, following the EDA's decisions
3. trains a first random forest and checks it on the calibration split
4. tunes the forest's settings with a randomized search over time-based folds
5. saves the best model

## 1. Libraries
Imports everything the notebook uses:
- **pandas / numpy:** loading and handling the data
- **scikit-learn:** the pipeline pieces (`Pipeline`, `ColumnTransformer`, `FunctionTransformer`, `OneHotEncoder`, `SimpleImputer`), the model (`RandomForestClassifier`), the tuning tool (`RandomizedSearchCV`) and the scoring metrics
- **time / joblib:** timing the search and saving the final model to a file

In [3]:
import numpy as np
import pandas as pd
import time
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    brier_score_loss
)

## 2. Loading the Data
Loads the two splits used for training and checking. **The test split is not loaded.**

- **`modeling_train.parquet`:** Jan 2023 – Oct 2024. 169,827 drive-windows, 8,087 failures. Healthy windows were down-sampled to 20 per failure, so failures are 1 in 21.
- **`modeling_calib.parquet`:** Nov – Dec 2024. 297,009 drive-windows, 328 failures, at the real failure rate of about 1 in 905.

For each split the cell creates:
- **`X_..._raw`:** every column except the answer
- **`y_...`:** the answer, `label` (1 = the drive fails in the next 30 days)
- **`meta_...`:** serial number, window number and model. These are **not** model inputs; they're used to build the time-based folds.

In [4]:
train = pd.read_parquet("../../modeling/modeling_train.parquet")
calib = pd.read_parquet("../../modeling/modeling_calib.parquet")

X_train_raw, y_train = train.drop(columns="label"), train["label"].astype(int)
X_calib_raw, y_calib = calib.drop(columns="label"), calib["label"].astype(int)

meta_train = train[["serial_number", "window", "model"]].reset_index(drop=True)
meta_calib = calib[["serial_number", "window", "model"]].reset_index(drop=True)

print("Training shape:", X_train_raw.shape)
print("Calibration shape:", X_calib_raw.shape)

display(X_train_raw.head())
display(X_calib_raw.head())

display(y_train.head())
display(y_calib.head())

Training shape: (170205, 93)
Calibration shape: (297879, 93)


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,043d3be09f4a0010,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30
1,0564f6f3fab90010,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30
2,1030A00KF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,5.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,696.0,30
3,1030A014F97G,0,100.0,100.0,0.0,0.0,100.0,100.0,4.0,5.0,...,-37.0,0.0,1.0,0.0,0.0,0.0,0.0,-1.0,696.0,30
4,1040A00BF97G,0,100.0,100.0,0.0,0.0,100.0,100.0,2.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-2.0,698.0,30


,serial_number,window,smart_10_normalized_first,smart_10_normalized_last,smart_10_raw_first,smart_10_raw_last,smart_12_normalized_first,smart_12_normalized_last,smart_12_raw_first,smart_12_raw_last,...,smart_3_raw_delta,smart_4_normalized_delta,smart_4_raw_delta,smart_5_normalized_delta,smart_5_raw_delta,smart_7_normalized_delta,smart_7_raw_delta,smart_9_normalized_delta,smart_9_raw_delta,drive_days_to_date
0,14A0A001F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,119.0,6
1,14B0A005F9RG,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,139.0,7
2,1PG6SGEV,22,100.0,100.0,0.0,0.0,100.0,100.0,8.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,78.0,5
3,1PG700ZV,22,100.0,100.0,0.0,0.0,100.0,100.0,10.0,10.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.0,5
4,1PG73SEV,22,100.0,100.0,0.0,0.0,100.0,100.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,171.0,8


0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64

## 3. Building the Model Inputs
Defines how the raw table is turned into model inputs. This only sets rules; nothing is calculated from the data here.

- **`KEEP_RAW`:** the SMART attributes kept, as raw values only. The EDA showed the vendor-normalized values lose detail and partly encode the brand. S10 is left out because it is always 0.
- **`vendor()`:** reads the manufacturer (Seagate, Toshiba, WDC, HGST) from the model name.
- **`build_features()`:**
  - keeps the first, last and change values for each kept attribute
  - keeps S9 (power-on hours) as drive age
  - turns the unknown capacity of -1 into a blank (23 training rows)
  - keeps days observed and the vendor
- **`NUMERIC` / `CATEGORICAL`:** the lists of number columns and the one text column (vendor).
- **`features` / `vendor_onehot`:** wrap these rules so they can sit inside the pipeline. The one-hot encoder turns vendor into yes/no columns, using a fixed vendor list so train and calibration always get the same columns.

In [5]:
KEEP_RAW = [1, 3, 4, 5, 7, 12, 192, 193, 194, 197, 198, 199]   # raw only; S9 -> age, S10 always 0
VENDORS = ["Seagate", "Toshiba", "WDC", "HGST", "Other"]


def vendor(model):
    m = str(model)
    if m.startswith("ST"): return "Seagate"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("WDC", "WUH")): return "WDC"
    if m.startswith(("HGST", "Hitachi")): return "HGST"
    return "Other"

## This Function Keeps raw SMART values only. Drops useless columns. Calculates drive age through power-on hours and cleans the storage column.
def build_features(df):
    X = pd.DataFrame(index=df.index)
    for i in KEEP_RAW:
        for s in ("first", "last", "delta"):
            X[f"s{i}_{s}"] = df[f"smart_{i}_raw_{s}"].astype("float64")
    X["s9_last"] = df["smart_9_raw_last"].astype("float64")          
    X["capacity_tb"] = (df["capacity_bytes"] / 1e12).round(0).where(df["capacity_bytes"] > 0)   # -1 = unknown -> NaN
    X["days_observed"] = df["days_observed"].astype("float64")
    X["vendor"] = df["model"].map(vendor)
    return X

NUMERIC = [f"s{i}_{s}" for i in KEEP_RAW for s in ("first", "last", "delta")] + \
          ["s9_last", "capacity_tb", "days_observed"]

CATEGORICAL = ["vendor"]

features = FunctionTransformer(build_features)
vendor_onehot = OneHotEncoder(categories=[VENDORS], handle_unknown="ignore", sparse_output=False)

## 4. Preprocessing Pipeline
Chains the preprocessing steps into one object, `rf_preprocess`:

1. **features:** applies `build_features` to the raw table
2. **columns:**
   - **numbers:** fills blanks with the **training** median and adds a "was missing" column for each column that had blanks. The blanks come from specific drive models, such as S197 on two Toshiba models, so the fact that a value is missing is itself useful.
   - **vendor:** one-hot encoded

Because this sits inside a pipeline, the medians are learned from training data only. They're then re-used on the calibration split, so no information leaks from calibration into training.

In [6]:
rf_preprocess = Pipeline([
    ("features", features),
    ("columns", ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median", add_indicator=True), NUMERIC),
        ("vendor", vendor_onehot, CATEGORICAL),
    ], verbose_feature_names_out=False)),
]).set_output(transform="pandas")

## 5. First Random Forest (Before Tuning)
Builds the full model, preprocessing followed by a random forest, with reasonable starting settings:

- **`n_estimators=300`:** 300 trees, whose votes are averaged
- **`max_features="sqrt"`:** each split looks at a random subset of columns, which makes the trees different from each other
- **`min_samples_leaf=2`:** each leaf must contain at least 2 windows, to limit memorizing
- **`class_weight="balanced_subsample"`:** gives failures more weight, recalculated for each tree's sample of the data
- **`random_state=42`:** makes results repeatable
- **`n_jobs=-1`:** uses all CPU threads

In [7]:
rf_model = Pipeline([
    ("preprocessing", rf_preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

### 5.1 Training the First Model
Fits the whole pipeline on the training split: the preprocessing learns its medians, then the forest grows its 300 trees.

In [8]:
rf_model.fit(X_train_raw, y_train)
joblib.dump(rf_model, "../artifacts/rf_base.joblib")

['../artifacts/rf_base.joblib']

### 5.2 Checking the First Model on the Calibration Split
Predicts a failure probability for every calibration window, then turns those probabilities into yes/no predictions at thresholds from 0.0 to 1.0. At each threshold it records:

- **PR-AUC (AP):** how well the model *ranks* failing drives above healthy ones, over all thresholds. This is our main metric, and random guessing scores 0.0011.
- **ROC-AUC:** another ranking score (0.5 = coin flip, 1.0 = perfect). It looks good easily when failures are rare, so it's secondary.
- **Precision:** of the drives flagged, the share that actually failed
- **Recall:** of the drives that failed, the share that were flagged
- **F1:** one number balancing precision and recall
- **Brier score:** how close the probabilities are to what actually happened (lower is better)

PR-AUC, ROC-AUC and the Brier score don't depend on the threshold, which is why they're the same in every row. Only precision, recall and F1 change.

**Result:**
- **Ranking:** PR-AUC is **0.0995**, against 0.0073 for the SMART 5 rule, 0.0011 for random and 0.082 for logistic regression. ROC-AUC is 0.884.
- **Thresholds:** at 0.9, precision is 17.6% and recall 20.1%, the best F1 (0.188). At the default 0.5, precision is only 5.0%. The probabilities are too high because training has 43× more failures than reality, so a threshold has to be chosen deliberately and the probabilities recalibrated later.

In [9]:
results=[]
y_prob = rf_model.predict_proba(X_calib_raw)[:, 1]

# Default decision threshold
for x in [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]:
    y_pred = (y_prob >= x).astype(int)

    results.append({
        "Threshold": x,
        "Model": "Random Forest",
        "PR-AUC (AP)": average_precision_score(y_calib, y_prob),
        "ROC-AUC": roc_auc_score(y_calib, y_prob),
        "Precision": precision_score(
            y_calib, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_calib, y_pred, zero_division=0
        ),
        "F1": f1_score(y_calib, y_pred, zero_division=0),
        "Brier Score": brier_score_loss(y_calib, y_prob)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

,Threshold,Model,PR-AUC (AP),ROC-AUC,Precision,Recall,F1,Brier Score
0,0.0,Random Forest,0.0923,0.8899,0.0011,1.0000,0.0022,0.0136
1,0.1,Random Forest,0.0923,0.8899,0.0074,0.7896,0.0147,0.0136
2,0.2,Random Forest,0.0923,0.8899,0.0173,0.7226,0.0337,0.0136
3,0.3,Random Forest,0.0923,0.8899,0.0289,0.6890,0.0555,0.0136
4,0.4,Random Forest,0.0923,0.8899,0.0378,0.6646,0.0715,0.0136
5,0.5,Random Forest,0.0923,0.8899,0.0474,0.6280,0.0882,0.0136
6,0.6,Random Forest,0.0923,0.8899,0.0590,0.5854,0.1072,0.0136
7,0.7,Random Forest,0.0923,0.8899,0.0747,0.5244,0.1307,0.0136
8,0.8,Random Forest,0.0923,0.8899,0.1020,0.4116,0.1634,0.0136
9,0.9,Random Forest,0.0923,0.8899,0.1589,0.2073,0.1799,0.0136


## 6. Tuning the Random Forest
### 6.1 Search Setup and Time-Based Folds
Sets up three things for tuning:

- **`rf_pipeline`:** the same preprocessing plus a forest whose settings will be searched (`class_weight="balanced"`)
- **`param_distributions`:** the settings to try:
  - number of trees
  - maximum tree depth
  - minimum windows needed to split a node, and to form a leaf
  - share of columns tried at each split
  - maximum number of leaves

  Together that's 864 combinations.
- **`time_folds()`:** builds 4 cross-validation folds **by time**. Each fold trains on earlier months and validates on later months, with a one-window gap in between, because each label looks one window ahead. This mirrors real use: predicting the future from the past.

The printout shows the 4 folds growing: training uses more months each time, and each validation block covers the next few months.

In [10]:
rf_pipeline = Pipeline([
    ("preprocess", rf_preprocess),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",   # failures are 1 in 21 in train: give them more weight
        n_jobs=-1,                 # use all 8 threads for each forest
        random_state=42,
    )),
])

param_distributions = {
    "classifier__n_estimators": [300, 400, 500, 600, 800],

    "classifier__max_depth": [4, 5, 6, 7, 8],

    "classifier__min_samples_split": [2, 3, 4, 5, 6, 8],

    "classifier__min_samples_leaf": [4, 5, 6, 7, 8, 10],

    "classifier__max_features": [0.35, 0.4, 0.5, 0.6, 0.7],

    "classifier__max_leaf_nodes": [350, 400, 450, 500, 550, 600],

    "classifier__bootstrap": [True],
}

## Creating a cross validation set.
def time_folds(meta, n_splits=4, gap=1):
    """Expanding-window folds by calendar window. Skips `gap` windows between train and validation,
    because a window's label looks one window ahead."""
    w = meta["window"].to_numpy()
    windows = np.sort(np.unique(w))
    blocks = np.array_split(windows, n_splits + 1)
    folds = []
    for k in range(1, n_splits + 1):
        valid_w = blocks[k]
        train_w = windows[windows < valid_w.min() - gap]
        folds.append((np.flatnonzero(np.isin(w, train_w)), np.flatnonzero(np.isin(w, valid_w))))
    return folds

cv_folds = time_folds(meta_train)
for k, (tr_idx, va_idx) in enumerate(cv_folds):
    print(f"fold {k}: train {len(tr_idx):,} rows -> validate {len(va_idx):,} rows")


fold 0: train 28,001 rows -> validate 37,252 rows
fold 1: train 64,564 rows -> validate 31,291 rows
fold 2: train 95,661 rows -> validate 32,971 rows
fold 3: train 128,294 rows -> validate 33,683 rows


### 6.2 Untuned Model on the Time Folds
Scores the untuned random forest from section 5 (`rf_model`) on the same 4 time-based folds the search uses, with the same four metrics: precision, recall, F1 and PR-AUC. This is the "before" number for tuning.

**Why this is needed.** Section 5.2 scored the untuned random forest on the calibration split, where failures are about 1 in 905. The search scores on training folds, where they are 1 in 21. PR-AUC depends heavily on the failure rate, so the 5.2 score (0.0995) can't be compared with the search scores in 6.3. Scoring the untuned random forest on the same folds the search uses leaves tuning as the only difference between them.

- **`cross_validate`:** trains the untuned model once per fold, scores it on that fold's validation months, and reports the mean and std of each metric over the 4 folds.
- **`scoring`:** the same four metrics the search uses, so the before and after numbers are measured the same way.
- **`n_jobs=1`:** runs one fold at a time, as the search does, because each forest already uses every CPU thread and several large forests at once could run out of memory.

**One difference to state in the comparison.** The untuned forest uses `class_weight="balanced_subsample"`, while every tuned setting uses `class_weight="balanced"` (set on `rf_pipeline` in 6.1). So the before-and-after gap reflects the change in class weighting as well as the settings searched.

**Result:** *to be filled in once the cell has run.*
- **Ranking:** mean PR-AUC over the 4 folds (std), and why it is far above the 0.0995 in section 5.2.
- **At the default 0.5 threshold:** precision, recall and F1, each with its std.
- **Time:** mean fit time per fold.

In [11]:
# cross_validate isn't in the section 1 imports, so it is imported here
from sklearn.model_selection import cross_validate

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "pr_auc": "average_precision",
}

# n_jobs=1: one fold at a time, as the search does; each model already uses every CPU thread
start = time.time()
cv_untuned_results = cross_validate(rf_model, X_train_raw, y_train, cv=cv_folds, scoring=scoring, n_jobs=1)
print(f"Cross-validation took {(time.time() - start) / 60:.1f} minutes")
print(" ")

for i in cv_untuned_results:
    print(f'{i} mean: {np.mean(cv_untuned_results[i])}')
    print(f'{i} standard deviation: {np.std(cv_untuned_results[i])}')
    print(' ')


Cross-validation took 0.9 minutes
 
fit_time mean: 13.353322625160217
fit_time standard deviation: 8.006848028397268
 
score_time mean: 0.8436192274093628
score_time standard deviation: 0.15207836527878382
 
test_precision mean: 0.6710241217950605
test_precision standard deviation: 0.033530712571542894
 
test_recall mean: 0.5426387381723681
test_recall standard deviation: 0.02974345748783955
 
test_f1 mean: 0.5999247619277599
test_f1 standard deviation: 0.030607738979005976
 
test_pr_auc mean: 0.5779266058858523
test_pr_auc standard deviation: 0.036532468868127214
 


### 6.3 Randomized Search, Results and Saving the Best Model
- **Randomized search:** tries **20 random combinations** from the 864. Each is trained once per fold, giving **80 fits**, and scored by PR-AUC (`average_precision`). `n_jobs=1` trains one forest at a time, because each forest already uses every CPU thread and several large forests at once could run out of the 12 GB of memory.
- **Refit:** the best combination is retrained on all of the training data (`refit=True`).
- **Results table:**
  - `mean_test_score`: the average PR-AUC over the 4 folds
  - `std_test_score`: how much the 4 fold scores vary
  - `rank_test_score`: 1 = best

  The table is saved to `rf_search_results_ranked.csv`.
- **Save:** the best model is saved to `rf_best.joblib`, so it can be reloaded without retraining.

**Result:**
- **Time:** the search took **36 minutes**.
- **Best settings:** **500 trees, no depth limit, `min_samples_leaf=10`, `min_samples_split=20`, `max_features="sqrt"`**, with a mean time-CV PR-AUC of **0.573** (std 0.032).
- **Close calls:** the top 6 settings are within 0.004 of each other, less than one std apart, so they perform about the same. The 6th-ranked setting (300 trees, depth 10) trains in about 7 seconds per fold against 17, which would be a faster alternative.

These scores come from training folds, where failures are 1 in 21, so they're much higher than PR-AUC on the calibration split. Use them only to compare settings with each other.

In [12]:
search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    scoring={
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "pr_auc": "average_precision",
    },
    cv=cv_folds,
    n_jobs=1,
    refit="pr_auc",
    random_state=42,
    verbose=2,
    return_train_score=False,
)

start = time.time()
search.fit(X_train_raw, y_train)
print(f"\nSearch took {(time.time() - start) / 60:.1f} minutes")

cv_results = pd.DataFrame(search.cv_results_)

# All four metrics, averaged across the CV folds (column names follow the scoring keys)
results = pd.DataFrame({
    "PR_AUC":         cv_results["mean_test_pr_auc"],
    "PR_AUC_std":     cv_results["std_test_pr_auc"],
    "Precision":      cv_results["mean_test_precision"],
    "Precision_std":  cv_results["std_test_precision"],
    "Recall":         cv_results["mean_test_recall"],
    "Recall_std":     cv_results["std_test_recall"],
    "F1":             cv_results["mean_test_f1"],
    "F1_std":         cv_results["std_test_f1"],
    "mean_fit_time":  cv_results["mean_fit_time"],
})

for col in cv_results.columns:
    if col.startswith("param_"):
        results[col] = cv_results[col]

# Rank by PR-AUC (the same ranking the search used to pick the best model)
results["PR_AUC_rank"] = cv_results["rank_test_pr_auc"]
results = results.sort_values("PR_AUC_rank").reset_index(drop=True)

results.to_csv("rf_search_results_ranked.csv", index=False)

# Save the best model (chosen by PR-AUC, refit on all of train)
best_rf = search.best_estimator_
joblib.dump(best_rf, "../artifacts/rf_best.joblib")


Fitting 4 folds for each of 20 candidates, totalling 80 fits
[CV] END classifier__bootstrap=True, classifier__max_depth=8, classifier__max_features=0.4, classifier__max_leaf_nodes=600, classifier__min_samples_leaf=6, classifier__min_samples_split=4, classifier__n_estimators=800; total time=  12.9s
[CV] END classifier__bootstrap=True, classifier__max_depth=8, classifier__max_features=0.4, classifier__max_leaf_nodes=600, classifier__min_samples_leaf=6, classifier__min_samples_split=4, classifier__n_estimators=800; total time=  29.8s
[CV] END classifier__bootstrap=True, classifier__max_depth=8, classifier__max_features=0.4, classifier__max_leaf_nodes=600, classifier__min_samples_leaf=6, classifier__min_samples_split=4, classifier__n_estimators=800; total time=  43.8s
[CV] END classifier__bootstrap=True, classifier__max_depth=8, classifier__max_features=0.4, classifier__max_leaf_nodes=600, classifier__min_samples_leaf=6, classifier__min_samples_split=4, classifier__n_estimators=800; total 

['../artifacts/rf_best.joblib']

In [13]:
print("Best PR-AUC (time-CV on train):", round(search.best_score_, 4))
print("Best parameters:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

print("\nTop 10 models ranked by PR-AUC:")
display(results.head(10))

Best PR-AUC (time-CV on train): 0.5748
Best parameters:
  classifier__n_estimators: 600
  classifier__min_samples_split: 3
  classifier__min_samples_leaf: 10
  classifier__max_leaf_nodes: 450
  classifier__max_features: 0.35
  classifier__max_depth: 8
  classifier__bootstrap: True

Top 10 models ranked by PR-AUC:


,PR_AUC,PR_AUC_std,Precision,Precision_std,Recall,Recall_std,F1,F1_std,mean_fit_time,param_classifier__n_estimators,param_classifier__min_samples_split,param_classifier__min_samples_leaf,param_classifier__max_leaf_nodes,param_classifier__max_features,param_classifier__max_depth,param_classifier__bootstrap,PR_AUC_rank
0,0.574778,0.034668,0.389277,0.045523,0.685226,0.034901,0.495549,0.043843,26.063160,600,3,10,450,0.35,8,True,1
1,0.573337,0.035260,0.386811,0.048334,0.687714,0.032187,0.494063,0.046087,29.312615,600,6,7,500,0.40,8,True,2
2,0.573241,0.035633,0.380060,0.058242,0.686720,0.030937,0.487681,0.054655,39.773952,500,8,8,600,0.70,8,True,3
3,0.572486,0.035275,0.388320,0.046366,0.686248,0.032790,0.495017,0.044450,36.125176,800,4,6,600,0.40,8,True,4
4,0.571818,0.035786,0.383641,0.057132,0.680411,0.029703,0.489079,0.053713,20.104750,300,3,10,600,0.70,7,True,5
5,0.570517,0.036078,0.391874,0.044833,0.683648,0.034532,0.497217,0.042772,21.966358,500,2,4,450,0.35,8,True,6
6,0.568323,0.035398,0.392375,0.032552,0.679411,0.035406,0.497146,0.034256,11.883643,300,6,7,500,0.35,7,True,7
7,0.568112,0.035042,0.389422,0.036248,0.671033,0.031974,0.492503,0.037155,17.613712,300,5,8,500,0.70,6,True,8
8,0.565182,0.035226,0.389964,0.034654,0.671730,0.031695,0.493157,0.035708,36.550391,600,2,4,600,0.70,6,True,9
9,0.564868,0.036161,0.393855,0.030579,0.670277,0.033868,0.495978,0.032798,32.525156,600,6,4,450,0.60,6,True,10


## 7. Final Scoring on the Calibration Split
Scores the tuned random forest from 6.3 **once** on the calibration split and saves its predictions for the model comparison. Nothing is tuned here.

- **Why calib:** the search chose its settings on training folds, where failures are 1 in 21. The calibration split (November to December 2024) has failures at the real rate of about 1 in 905, comes after every training month, and was never used to fit or tune this model. It shows how the tuned model ranks drives under conditions like real use, and every model is scored on exactly the same rows, so the scores can be compared directly.
- **Why only once:** calib holds only 328 failures. Scoring it repeatedly and adjusting the model each time would tune the model to those 328 drives. The 2025 test split stays closed until the final model is chosen.
- **The model:** loaded from `../artifacts/rf_best.joblib`, the same file the final test will load, so the search doesn't need to be re-run.
- **Saved file:** `rf_calib_preds.csv` in this folder, one row per calibration window, with the columns `serial_number`, `window`, `label` and `prob`. The comparison notebook builds every model's PR-AUC and replacement-budget results from these files, and calibrates the winning model from them.

**Result:** *to be filled in once the cell has run.*
- **Ranking:** PR-AUC on calib, against 0.0995 for the untuned random forest in section 5.2. Both are on the calibration split, so this is a direct before-and-after comparison.
- **Saved:** row count of `rf_calib_preds.csv` (297,009 expected) and its number of failures (328 expected).

In [14]:
rf_final = joblib.load("../artifacts/rf_best.joblib")

calib_scores = rf_final.predict_proba(X_calib_raw)[:, 1]

calib_preds = pd.DataFrame({
    'serial_number': meta_calib['serial_number'].copy(),
    'window': meta_calib['window'].copy(),
    'label': y_calib.to_numpy(),
    'prob': calib_scores
})

calib_preds.to_csv('rf_calib_preds.csv', index = False)

print(f"Average precision score: {average_precision_score(calib_preds['label'], calib_preds['prob'])}")
print(f'Row count: {len(calib_preds)}')
print(f"Failure count: {calib_preds['label'].sum()}")


Average precision score: 0.08687338155521333
Row count: 297879
Failure count: 328
